In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [2]:
DATA_PATH = os.path.join("../data/", "santander-customer-satisfaction")

train_df = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))

print(f"Train 데이터 크기: {train_df.shape}")

train_df.head()

Train 데이터 크기: (76020, 371)


,ID,var3,var15,imp_ent_var16_ult1,imp_op_var39_comer_ult1,imp_op_var39_comer_ult3,imp_op_var40_comer_ult1,imp_op_var40_comer_ult3,imp_op_var40_efect_ult1,imp_op_var40_efect_ult3,...,saldo_medio_var33_hace2,saldo_medio_var33_hace3,saldo_medio_var33_ult1,saldo_medio_var33_ult3,saldo_medio_var44_hace2,saldo_medio_var44_hace3,saldo_medio_var44_ult1,saldo_medio_var44_ult3,var38,TARGET
0,1,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,39205.170000,0
1,3,2,34,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,49278.030000,0
2,4,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,67333.770000,0
3,8,2,37,0.0,195.0,195.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,64007.970000,0
4,10,2,39,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,117310.979016,0


In [3]:
# 정답 라벨 분리 (맨 마지막 컬럼 TARGET)
y_labels = train_df.iloc[:, -1].copy()

# ID와 TARGET을 제외한 순수 피처 세트 분리
X_features = train_df.drop(columns=['ID', 'TARGET']).copy()

print(f"X_features Shape: {X_features.shape}")
print(f"y_labels Shape: {y_labels.shape}")

X_features Shape: (76020, 369)
y_labels Shape: (76020,)


In [4]:
# var3 결측성 이상치(-999999)를 최빈값(2)으로 치환
X_features['var3'] = X_features['var3'].replace(-999999, 2)

print("var3 이상치 치환 완료 (-999999 잔여 개수):", (X_features['var3'] == -999999).sum())

var3 이상치 치환 완료 (-999999 잔여 개수): 0


In [5]:
# RandomForest는 트리 분할 시 임계값(threshold) 기준으로만 동작하기 때문에
# 단조 변환(log 등)을 적용해도 분할 결과나 성능이 달라지지 않습니다.
# 즉 RF 파이프라인에서는 불필요한 단계라 원본 코드는 참고용으로 주석 처리해 남겨둡니다.

# X_features['var38'] = np.log1p(X_features['var38'])
# print("var38 로그 변환 완료")

print("var38 로그 변환 생략 (RandomForest는 단조 변환에 불변하여 불필요)")

var38 로그 변환 생략 (RandomForest는 단조 변환에 불변하여 불필요)


In [6]:
# 고객별 0의 개수 카운트
X_features['n0'] = (X_features == 0).sum(axis=1)

# 고객별 거래/잔액의 표준편차
X_features['row_std'] = X_features.std(axis=1)

# var38 최빈값(결측치를 평균으로 대체한 자리표시자로 추정) 플래그 추가
var38_mode = X_features['var38'].value_counts().idxmax()
X_features['var38_is_missing'] = (X_features['var38'] == var38_mode).astype(int)
print(f"var38 최빈값({var38_mode}) 플래그(var38_is_missing) 생성 완료 (해당 건수: {X_features['var38_is_missing'].sum()})")

print("행 통계량 파생 변수(n0, row_std) 생성 완료")

var38 최빈값(117310.979016494) 플래그(var38_is_missing) 생성 완료 (해당 건수: 14868)
행 통계량 파생 변수(n0, row_std) 생성 완료


In [7]:
# [주의] 이 시점(train/test 분할 이전)에 상수/중복 컬럼을 제거하면
# 테스트 세트의 피처 분포 정보가 전처리 결정에 섞여 들어가는 문제가 있습니다(train/test 정보 누수).
# 분할 후 X_tr 기준으로만 다시 판단하도록 아래(분할 이후) 셀로 로직을 옮겼습니다.
# 원본 코드는 참고용으로 주석 처리하여 남겨둡니다.

# zero_var_cols = [col for col in X_features.columns if X_features[col].nunique() == 1]
# X_features.drop(columns=zero_var_cols, inplace=True)
# print(f"1) 제거된 상수 컬럼 수: {len(zero_var_cols)}")

# dup_cols = X_features.T.duplicated()
# dup_col_names = X_features.columns[dup_cols].tolist()
# X_features.drop(columns=dup_col_names, inplace=True)
# print(f"2) 제거된 중복 컬럼 수: {len(dup_col_names)}")

# manual_remove = [c for c in X_features.columns if 'var6' in c] + [
#     'delta_imp_reemb_var13_1y3', 'delta_imp_reemb_var17_1y3',
#     'delta_imp_trasp_var17_in_1y3', 'delta_imp_trasp_var33_in_1y3'
# ]
# manual_remove = [c for c in manual_remove if c in X_features.columns]
# X_features.drop(columns=manual_remove, inplace=True)
# print(f"3) 추가 제거된 유사/노이즈 컬럼 수: {len(manual_remove)}")

print("상수/중복/수동 제거 컬럼 판단은 분할 이후 X_tr 기준으로 아래 셀에서 다시 수행합니다.")

상수/중복/수동 제거 컬럼 판단은 분할 이후 X_tr 기준으로 아래 셀에서 다시 수행합니다.


In [8]:
# 1차 분할: 학습 세트(80%)와 최종 테스트 세트(20%)로 분리
X_train, X_test, y_train, y_test = train_test_split(
    X_features, y_labels,
    test_size=0.2, 
    stratify=y_labels, 
    random_state=0
)

# 2차 분할: 학습 세트를 다시 훈련용(70%)과 조기 중단 감시용(30%)으로 분리
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.3, 
    stratify=y_train, 
    random_state=0
)

print(f"훈련 세트(X_tr) Shape: {X_tr.shape}")
print(f"검증 세트(X_val) Shape: {X_val.shape}")
print(f"테스트 세트(X_test) Shape: {X_test.shape}")

훈련 세트(X_tr) Shape: (42571, 372)
검증 세트(X_val) Shape: (18245, 372)
테스트 세트(X_test) Shape: (15204, 372)


In [9]:
from sklearn.ensemble import RandomForestClassifier

# ===== (0) 상수/중복/수동 노이즈 컬럼 제거 =====
# train/test 정보 누수를 막기 위해 분할 이후 X_tr 기준으로만 판단하고,
# 동일한 컬럼 목록을 X_val, X_test에도 그대로 적용합니다.
zero_var_cols = [c for c in X_tr.columns if X_tr[c].nunique() == 1]
X_tr = X_tr.drop(columns=zero_var_cols)
X_val = X_val.drop(columns=zero_var_cols)
X_test = X_test.drop(columns=zero_var_cols)
print(f"1) 제거된 상수 컬럼 수: {len(zero_var_cols)}")

dup_col_names = X_tr.columns[X_tr.T.duplicated()].tolist()
X_tr = X_tr.drop(columns=dup_col_names)
X_val = X_val.drop(columns=dup_col_names)
X_test = X_test.drop(columns=dup_col_names)
print(f"2) 제거된 중복 컬럼 수: {len(dup_col_names)}")

manual_remove = [c for c in X_tr.columns if 'var6' in c] + [
    'delta_imp_reemb_var13_1y3', 'delta_imp_reemb_var17_1y3',
    'delta_imp_trasp_var17_in_1y3', 'delta_imp_trasp_var33_in_1y3'
]
manual_remove = [c for c in manual_remove if c in X_tr.columns]
X_tr = X_tr.drop(columns=manual_remove)
X_val = X_val.drop(columns=manual_remove)
X_test = X_test.drop(columns=manual_remove)
print(f"3) 추가 제거된 유사/노이즈 컬럼 수: {len(manual_remove)}")

print(f"컬럼 정리 후 훈련 세트(X_tr) Shape: {X_tr.shape}")
print(f"컬럼 정리 후 검증 세트(X_val) Shape: {X_val.shape}")
print(f"컬럼 정리 후 테스트 세트(X_test) Shape: {X_test.shape}")

# ===== (1) 피처 중요도 기반 컬럼 제거 =====
# 실제로 사용할 모델(RandomForest) 기준으로 중요도를 측정합니다.
# XGBoost 기준으로 중요도 0이라고 해서 RandomForest에서도 0이라는 보장이 없기 때문입니다.
base_rf = RandomForestClassifier(
    n_estimators=200, max_depth=10, min_samples_leaf=8,
    min_samples_split=8, random_state=0, n_jobs=-1
)
base_rf.fit(X_tr, y_tr)
feat_imp = pd.Series(base_rf.feature_importances_, index=X_tr.columns)

# --- 원본(XGBoost 기준) 코드는 참고용으로 주석 처리 ---
# base_xgb = XGBClassifier(
#     n_estimators=100,
#     learning_rate=0.1,
#     random_state=156,
#     n_jobs=-1
# )
# base_xgb.fit(X_tr, y_tr)
# feat_imp = pd.Series(base_xgb.feature_importances_, index=X_tr.columns)

zero_imp_cols = feat_imp[feat_imp == 0].index.tolist()
X_tr = X_tr.drop(columns=zero_imp_cols)
X_val = X_val.drop(columns=zero_imp_cols)
X_test = X_test.drop(columns=zero_imp_cols)
print(f"제거된 중요도 0인 컬럼 수 (RandomForest 기준): {len(zero_imp_cols)}")

# ===== (2) 스케일링 -> RandomForest에는 불필요하여 생략 =====
# 트리 기반 모델은 피처의 절대적 크기가 아니라 분할 임계값만 사용하므로
# StandardScaler로 평균/분산을 맞춰도 트리 구조나 성능이 달라지지 않습니다.
# 원본 코드는 참고용으로 주석 처리하여 남겨둡니다.

# from sklearn.preprocessing import StandardScaler
# scaler = StandardScaler()
# X_tr_scaled = scaler.fit_transform(X_tr)
# X_val_scaled = scaler.transform(X_val)
# X_test_scaled = scaler.transform(X_test)

# ===== (3) PCA -> RandomForest에는 기본적으로 불필요하여 생략 =====
# RandomForest는 상관되거나 중복된 피처가 있어도 분할 과정에서 알아서 걸러 쓰기 때문에
# PCA로 얻는 이득이 크지 않은 경우가 많고, 오히려 해석 가능한 분할 기준을 흐릴 수 있습니다.
# 필요하다면(예: 실험적으로 검증 AUC 개선이 확인되는 경우) 아래 원본 코드를 다시 활성화하세요.
# 단, PCA를 쓰려면 위 StandardScaler 주석도 함께 해제해서 스케일링을 먼저 해야 합니다.

# from sklearn.decomposition import PCA
# pca = PCA(n_components=5, random_state=156)
# pca_tr = pca.fit_transform(X_tr_scaled)
# pca_val = pca.transform(X_val_scaled)
# pca_test = pca.transform(X_test_scaled)
#
# X_tr['pca1'] = pca_tr[:, 0]
# X_tr['pca2'] = pca_tr[:, 1]
# X_tr['pca3'] = pca_tr[:, 2]
# X_tr['pca4'] = pca_tr[:, 3]
# X_tr['pca5'] = pca_tr[:, 4]
#
# X_val['pca1'] = pca_val[:, 0]
# X_val['pca2'] = pca_val[:, 1]
# X_val['pca3'] = pca_val[:, 2]
# X_val['pca4'] = pca_val[:, 3]
# X_val['pca5'] = pca_val[:, 4]
#
# X_test['pca1'] = pca_test[:, 0]
# X_test['pca2'] = pca_test[:, 1]
# X_test['pca3'] = pca_test[:, 2]
# X_test['pca4'] = pca_test[:, 3]
# X_test['pca5'] = pca_test[:, 4]

print(f"훈련 세트(X_tr) 최종 Shape: {X_tr.shape}")
print(f"검증 세트(X_val) 최종 Shape: {X_val.shape}")
print(f"테스트 세트(X_test) 최종 Shape: {X_test.shape}")

1) 제거된 상수 컬럼 수: 40


2) 제거된 중복 컬럼 수: 29
3) 추가 제거된 유사/노이즈 컬럼 수: 9
컬럼 정리 후 훈련 세트(X_tr) Shape: (42571, 294)
컬럼 정리 후 검증 세트(X_val) Shape: (18245, 294)
컬럼 정리 후 테스트 세트(X_test) Shape: (15204, 294)


제거된 중요도 0인 컬럼 수 (RandomForest 기준): 75
훈련 세트(X_tr) 최종 Shape: (42571, 219)
검증 세트(X_val) 최종 Shape: (18245, 219)
테스트 세트(X_test) 최종 Shape: (15204, 219)


In [10]:
from sklearn.ensemble import RandomForestClassifier

# 랜덤 포레스트 n_estimators(트리 개수)별 검증 세트 AUC 비교 -> 최적값 탐색
# class_weight='balanced' 추가: TARGET이 3.96%밖에 안 되는 불균형 데이터라 소수 클래스 가중치를 보정
n_estimators_list = [100, 200, 300, 400, 500]
val_auc_scores = []

for n in n_estimators_list:
    rf = RandomForestClassifier(
        n_estimators=n, max_depth=10, min_samples_leaf=8,
        min_samples_split=8, class_weight='balanced', random_state=0, n_jobs=-1
    )
    rf.fit(X_tr, y_tr)
    val_pred = rf.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, val_pred)
    val_auc_scores.append(auc)
    print(f"n_estimators={n}: 검증 AUC={auc:.4f}")

best_n = n_estimators_list[np.argmax(val_auc_scores)]
print(f"\n최적 n_estimators: {best_n} (검증 AUC={max(val_auc_scores):.4f})")

# 최적 n_estimators로 최종 테스트 세트 AUC 산출
rf_best = RandomForestClassifier(
    n_estimators=best_n, max_depth=10, min_samples_leaf=8,
    min_samples_split=8, class_weight='balanced', random_state=0, n_jobs=-1
)
rf_best.fit(X_tr, y_tr)
test_pred = rf_best.predict_proba(X_test)[:, 1]
rf_roc_score = roc_auc_score(y_test, test_pred)

print("=" * 40)
print(f"랜덤 포레스트 최종 테스트 세트 ROC-AUC: {rf_roc_score:.4f}")
print("=" * 40)

n_estimators=100: 검증 AUC=0.8269


n_estimators=200: 검증 AUC=0.8269


n_estimators=300: 검증 AUC=0.8278


n_estimators=400: 검증 AUC=0.8275


n_estimators=500: 검증 AUC=0.8269

최적 n_estimators: 300 (검증 AUC=0.8278)


랜덤 포레스트 최종 테스트 세트 ROC-AUC: 0.7954


In [11]:
from sklearn.model_selection import RandomizedSearchCV, PredefinedSplit
from scipy.stats import randint

# X_tr(훈련)는 탐색용 학습에만, X_val(검증)은 평가에만 쓰이도록 PredefinedSplit 구성
X_search = pd.concat([X_tr, X_val], axis=0)
y_search = pd.concat([y_tr, y_val], axis=0)
test_fold = [-1] * len(X_tr) + [0] * len(X_val)
ps = PredefinedSplit(test_fold)

# class_weight 탐색 추가: 불균형 데이터(TARGET 3.96%) 보정 여부까지 함께 탐색
param_dist = {
    'n_estimators': randint(100, 600),
    'max_depth': [6, 8, 10, 12, 15, 20, None],
    'min_samples_leaf': randint(1, 15),
    'min_samples_split': randint(2, 20),
    'max_features': ['sqrt', 'log2', 0.5, None],
    'class_weight': ['balanced', 'balanced_subsample', None],
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=0, n_jobs=1),
    param_distributions=param_dist,
    n_iter=100,
    scoring='roc_auc',
    cv=ps,
    random_state=42,
    n_jobs=-1,
    # [주의] refit 기본값은 True이며, 이 경우 최적 파라미터를 찾은 뒤
    # X_search(=X_tr+X_val) 전체로 최종 모델을 다시 학습시킵니다.
    # 그러면 X_tr만으로 학습한 다른 모델들과 학습 데이터 양이 달라져 공정한 비교가 되지 않으므로
    # refit=False로 두고 아래에서 X_tr만으로 직접 재학습합니다.
    refit=False,
)
rf_search.fit(X_search, y_search)

print("최적 하이퍼파라미터:", rf_search.best_params_)
print(f"검증 AUC: {rf_search.best_score_:.4f}")

# --- 원본 코드(자동 refit에 의존, X_tr+X_val로 재학습됨)는 참고용으로 주석 처리 ---
# best_rf = rf_search.best_estimator_

# X_tr만으로 명시적으로 재학습 (X_val은 계속 평가 전용으로 유지)
best_rf = RandomForestClassifier(
    **rf_search.best_params_,
    random_state=0,
    n_jobs=-1,
)
best_rf.fit(X_tr, y_tr)

# 최종 테스트 세트 AUC 산출
test_pred = best_rf.predict_proba(X_test)[:, 1]
rf_roc_score = roc_auc_score(y_test, test_pred)

print("=" * 40)
print(f"랜덤 포레스트(튜닝) 최종 테스트 세트 ROC-AUC: {rf_roc_score:.4f}")
print("=" * 40)

최적 하이퍼파라미터: {'class_weight': None, 'max_depth': 8, 'max_features': 0.5, 'min_samples_leaf': 7, 'min_samples_split': 18, 'n_estimators': 189}
검증 AUC: 0.8444


랜덤 포레스트(튜닝) 최종 테스트 세트 ROC-AUC: 0.8231
